# UMAP — Uniform Manifold Approximation and Projection

## 1. Definition and intuition
UMAP builds a fuzzy graph of local neighbor relationships in the original space, then finds a low-dimensional embedding whose fuzzy neighborhood graph resembles the high-dimensional graph. It can be useful for visualization and, unlike standard scikit-learn t-SNE, common UMAP implementations also support transforming new observations.

## 2. Core mathematics, conceptually
For each point, UMAP estimates a local connectivity offset $\rho_i$ and scale $\sigma_i$, then converts neighbor distances into fuzzy memberships similar to
$$
p_{j|i}=\exp\left(-\frac{\max(0,d(x_i,x_j)-\rho_i)}{\sigma_i}\right).
$$
Directed memberships are combined with a fuzzy union, often $p_{ij}+p_{ji}-p_{ij}p_{ji}$. The low-dimensional graph uses a smooth curve $q(d)=1/(1+a d^{2b})$, with parameters related to `min_dist` and `spread`; stochastic optimization approximates a cross-entropy objective between the two graphs.

UMAP is based on a richer theoretical construction involving fuzzy simplicial sets. These equations are a practical intuition, not the complete derivation.

## 3. Why it works and assumptions
The method preserves neighborhood structure encoded by the graph. Its results depend strongly on the neighbor graph, distance metric, `n_neighbors`, `min_dist`, and stochastic optimization. UMAP embeddings can distort global distances and apparent cluster size; visualization should not be used as a standalone statistical test.

In [1]:
import importlib.util
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

SEED = 42
digits = load_digits()
X_train, X_test, y_train, y_test = train_test_split(
    digits.data, digits.target, test_size=0.18, stratify=digits.target, random_state=SEED
)
scaler = StandardScaler()
X_small = scaler.fit_transform(X_train[:500])
y_small = y_train[:500]
print("UMAP sample:", X_small.shape, "| umap installed:", importlib.util.find_spec("umap") is not None)


UMAP sample: (500, 64) | umap installed: False


## 4. From-scratch fuzzy neighborhood graph (educational component)

This cell implements a simplified local fuzzy graph with nearest neighbors, local-connectivity offset, and local scale search. It is **not a complete UMAP implementation**: UMAP's full low-dimensional cross-entropy optimizer, negative sampling, and implementation-specific details are left to the mature library. The explicit separation prevents conflating a graph-building component with the complete algorithm.

In [2]:
def fuzzy_knn_graph_scratch(X, n_neighbors=15, local_connectivity=1.0, n_iter=40):
    X = np.asarray(X, dtype=float)
    if X.ndim != 2 or len(X) <= n_neighbors or n_neighbors < 2:
        raise ValueError("Invalid X or n_neighbors")
    distances, indices = NearestNeighbors(n_neighbors=n_neighbors+1).fit(X).kneighbors(X)
    distances, indices = distances[:,1:], indices[:,1:]
    n = len(X)
    rows, cols, vals = [], [], []
    target = np.log2(n_neighbors)
    for i in range(n):
        di = distances[i]
        nonzero = di[di > 0]
        rho = nonzero[min(int(local_connectivity)-1, len(nonzero)-1)] if len(nonzero) else 0.0
        lo, hi, sigma = 0.0, np.inf, 1.0
        for _ in range(n_iter):
            membership_sum = np.exp(-np.maximum(0, di-rho)/max(sigma, 1e-12)).sum()
            if abs(membership_sum-target) < 1e-3:
                break
            if membership_sum > target:
                hi = sigma
                sigma = (lo+hi)/2
            else:
                lo = sigma
                sigma = sigma*2 if np.isinf(hi) else (lo+hi)/2
        weights = np.exp(-np.maximum(0, di-rho)/max(sigma, 1e-12))
        rows.extend([i]*n_neighbors)
        cols.extend(indices[i].tolist())
        vals.extend(weights.tolist())
    from scipy.sparse import coo_matrix
    directed = coo_matrix((vals,(rows,cols)), shape=(n,n)).tocsr()
    # Fuzzy union: A + A.T - A elementwise-multiplied by A.T.
    graph = directed + directed.T - directed.multiply(directed.T)
    graph.setdiag(0)
    graph.eliminate_zeros()
    return graph, indices, distances

fuzzy_graph, knn_indices, knn_distances = fuzzy_knn_graph_scratch(X_small, n_neighbors=15)
print("Fuzzy graph:", fuzzy_graph.shape, "edges:", fuzzy_graph.nnz, "density:", round(fuzzy_graph.nnz/(len(X_small)**2), 4))
assert np.all(fuzzy_graph.data >= 0)
assert (fuzzy_graph - fuzzy_graph.T).nnz == 0


Fuzzy graph: (500, 500) edges: 10372 density: 0.0415


## 5. Use the full UMAP implementation (optional dependency)

Install `umap-learn` if needed. This notebook runs without it: the graph-construction cell above still demonstrates a key UMAP component. If installed, the next cell performs the actual embedding with the library. Use the `transform` method for new data when supported by the fitted UMAP model.

In [3]:
if importlib.util.find_spec("umap") is not None:
    import umap
    t0 = time.perf_counter()
    reducer = umap.UMAP(
        n_neighbors=15, min_dist=0.1, n_components=2,
        metric="euclidean", random_state=SEED
    )
    embedding = reducer.fit_transform(X_small)
    elapsed = time.perf_counter() - t0
    print("UMAP fit seconds:", round(elapsed, 3), "embedding:", embedding.shape)
    plt.scatter(embedding[:,0], embedding[:,1], c=y_small, cmap="tab10", s=13)
    plt.title("UMAP embedding"); plt.colorbar(label="Digit label"); plt.show()
    assert np.isfinite(embedding).all()
else:
    print("umap-learn is not installed. To run the full embedding locally: pip install umap-learn")


umap-learn is not installed. To run the full embedding locally: pip install umap-learn


## 6. Hyperparameters, complexity, scaling and GPU
- `n_neighbors`: local-vs-broader graph scale; lower values focus on local structure.
- `min_dist`: minimum spacing in the embedding; lower values often permit tighter clumps but should not be overinterpreted.
- `metric`: defines input-space neighbor geometry.
- `n_components`: embedding dimension; `random_state` aids reproducibility.
- Neighbor search and graph storage are commonly more scalable than dense all-pairs distances, but exact complexity depends on neighbor-index method and implementation. Memory is driven by input, sparse graph, and optimization state.
- Millions of samples: use approximate neighbor search, reduce input dimension if appropriate, monitor graph memory, and use batch/transform workflows when available.
- Standard CPU `umap-learn` is common; RAPIDS cuML offers GPU UMAP in supported environments. GPU support is external to scikit-learn and depends on libraries/hardware.

## 7. Best practices and synthesis
UMAP is a nonlinear graph-based embedding. Use several `n_neighbors` / `min_dist` settings, compare against PCA and t-SNE, and validate neighborhood preservation or downstream tasks when possible. Do not treat visual islands as statistically proven clusters.

In [4]:
assert fuzzy_graph.shape == (len(X_small), len(X_small))
assert np.isfinite(fuzzy_graph.data).all()
assert np.all(fuzzy_graph.data <= 1.0 + 1e-7)
try:
    fuzzy_knn_graph_scratch(np.array([1,2,3]), 2)
except ValueError:
    pass
else:
    raise AssertionError("Expected invalid input shape to raise")
print("UMAP fuzzy-graph checks passed.")


UMAP fuzzy-graph checks passed.
